# 05 Final Load Prep

Prepare the final analysis dataset, compute reusable KPI tables, and export the Tableau-ready files that will power the dashboard.

In [1]:
from pathlib import Path

import pandas as pd
from IPython.display import display

PROJECT_ROOT = Path.cwd().resolve().parent if Path.cwd().resolve().name == 'notebooks' else Path.cwd().resolve()
DATA_PATH = PROJECT_ROOT / 'data/processed/cleaned_dataset.csv'
TABLEAU_READY_PATH = PROJECT_ROOT / 'data/processed/tableau_ready_dataset.csv'
KPI_PATH = PROJECT_ROOT / 'data/processed/tableau_kpi_summary.csv'

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 120)

final_df = pd.read_csv(DATA_PATH)
final_df.head()

,age,job,marital,education,default,balance,housing,loan,contact,day,month,duration,campaign,pdays,previous,poutcome,y,previously_contacted,duration_minutes,age_group,balance_band,campaign_band
0,58,management,married,tertiary,no,2143,yes,no,unknown,5,may,261,1,-1,0,unknown,0,0,4.35,55-64,medium,1
1,44,technician,single,secondary,no,29,yes,no,unknown,5,may,151,1,-1,0,unknown,0,0,2.52,35-44,low,1
2,33,entrepreneur,married,secondary,no,2,yes,yes,unknown,5,may,76,1,-1,0,unknown,0,0,1.27,25-34,low,1
3,47,blue-collar,married,unknown,no,1506,yes,no,unknown,5,may,92,1,-1,0,unknown,0,0,1.53,45-54,medium,1
4,33,unknown,single,unknown,no,1,no,no,unknown,5,may,198,1,-1,0,unknown,0,0,3.30,25-34,low,1


In [7]:
kpi_overview = pd.DataFrame(
    {
        'kpi': [
            'total_contacts',
            'total_subscriptions',
            'subscription_rate',
            'average_age',
            'average_balance',
            'median_campaign_count',
            'previously_contacted_share',
        ],
        'value': [
            len(final_df),
            int(final_df['y'].sum()),
            final_df['y'].mean(),
            final_df['age'].mean(),
            final_df['balance'].mean(),
            final_df['campaign'].median(),
            final_df['previously_contacted'].mean(),
        ],
    }
)

monthly_kpi = (
    final_df.groupby(['month'], dropna=False)
    .agg(
        total_contacts=('y', 'size'),
        subscriptions=('y', 'sum'),
        subscription_rate=('y', 'mean'),
    )
    .reset_index()
    .sort_values('month')
)

segment_kpi = (
    final_df.groupby(['job', 'education'], dropna=False)
    .agg(
        total_contacts=('y', 'size'),
        subscriptions=('y', 'sum'),
        subscription_rate=('y', 'mean'),
    )
    .reset_index()
    .sort_values('subscription_rate', ascending=False)
)

display(kpi_overview)
display(monthly_kpi)
display(segment_kpi.head(15))


,kpi,value
0,total_contacts,45211.000000
1,total_subscriptions,5289.000000
2,subscription_rate,0.116985
3,average_age,40.936210
4,average_balance,1362.272058
5,median_campaign_count,2.000000
6,previously_contacted_share,0.182633


,month,total_contacts,subscriptions,subscription_rate
0,apr,2932,577,0.196794
1,aug,6247,688,0.110133
2,dec,214,100,0.467290
3,feb,2649,441,0.166478
4,jan,1403,142,0.101212
5,jul,6895,627,0.090935
6,jun,5341,546,0.102228
7,mar,477,248,0.519916
8,may,13766,925,0.067195
9,nov,3970,403,0.101511


,job,education,total_contacts,subscriptions,subscription_rate
32,student,primary,44,16,0.363636
33,student,secondary,508,151,0.297244
22,retired,tertiary,366,101,0.275956
34,student,tertiary,223,59,0.264574
35,student,unknown,163,43,0.263804
23,retired,unknown,119,30,0.252101
20,retired,primary,795,178,0.223899
21,retired,secondary,984,207,0.210366
19,management,unknown,242,48,0.198347
42,unemployed,tertiary,289,56,0.193772


In [8]:
   
print(f'Final row count: {len(tableau_ready_df):,}')
print(f'Final column count: {len(tableau_ready_df.columns):,}')


Final row count: 45,211
Final column count: 22
